In [ ]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload

In [ ]:
%autoreload
import torch
import pandas as pd
import torch.nn as nn
from pytorch_pretrained_bert import GPT2Tokenizer, BertAdam
from torch.utils.data import DataLoader
from torch.nn import BCEWithLogitsLoss, MSELoss
from source.gpt.data import Data
from source.gpt.model import GPT
from source.gpt.train import trainModel

In [ ]:
def loadModel(model, path):
    results = torch.load(path)
    model.load_state_dict(results['model_state_dict'])
    loss = results['loss']
    print('Model Loaded:', 'Loss:', loss)
    return model, loss

In [ ]:
def customLoss(preds, label, weight):
    bce_loss = BCEWithLogitsLoss(weight=weight)(preds.squeeze(), label.squeeze())
    return bce_loss

# def customLoss(preds, label, weight):
#     mse_loss = MSELoss(reduction='none')(preds.squeeze(), label.squeeze())
#     mse_loss = torch.mul(weight, mse_loss)
#     return mse_loss.mean()

In [ ]:
def train_model(device, fold, model):
    train_data = pd.read_csv('../../data/train_data_{}.csv'.format(fold))
    valid_data = pd.read_csv('../../data/valid_data_{}.csv'.format(fold))
    print('Dataset:',train_data.shape, valid_data.shape)
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    train_data = Data(tokenizer, train_data, True)
    valid_data = Data(tokenizer, valid_data, True)
    params = {}
    params['num_workers'] = 4
    params['pin_memory'] = True
    params['drop_last'] = True
    train_loader = DataLoader(dataset=train_data, batch_size=12, shuffle=True, **params)
    valid_loader = DataLoader(dataset=valid_data, batch_size=5, shuffle=False, **params)
    model = GPT().float().to(device)
    no_decay = ['bias', '.ln']
    param_optimizer = list(model.named_parameters())
    optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = BertAdam(optimizer_grouped_parameters, lr=2e-5, warmup=0.05, t_total=35000)
    params = {}
    params['train'] = train_loader
    params['valid'] = valid_loader
    params['device'] = device
    params['model'] = model
    params['optimizer'] = optimizer
    params['loss_fn'] = customLoss
    params['save'] = '../../model/gpt/fold-{}/'.format(fold)
    params['batch'] = 12
    trainModel(**params)
    return None

In [ ]:
train_model('cuda:0', 1, 'bert-base-uncased')

In [ ]:
train_model('cuda:0', 2, 'bert-base-uncased')

In [ ]:
train_model('cuda:0', 3, 'bert-base-uncased')